In [54]:
import networkx as nx
import pickle
import copy
from tqdm import tqdm

In [2]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    print(f"Load graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

def save_graph(filepath, G):
    with open(filepath, 'wb') as f:
        pickle.dump(G,f)
    print(f"Save graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges to {filepath}")

## Normalize all Base KGs 

In [18]:
healthy_kg = load_graph("../data/KG/hamonized_healthy_aging_kg.pkl")
ppi_new = load_graph("../datasets/base_kgs/ppi_new.pkl")
ppi_old = load_graph("../datasets/base_kgs/oldcleaned_ppi_kg.pkl")
prime_kg = load_graph("../datasets/base_kgs/prime_ad_kg.pkl")
ad_kg = load_graph("../datasets/base_kgs/ad_kg_reversed_noncausal_removed.pkl")

Load graph with 4161 nodes and 6956 edges
Load graph with 17042 nodes and 382526 edges
Load graph with 8910 nodes and 209730 edges
Load graph with 4909 nodes and 17542 edges
Load graph with 3732 nodes and 10554 edges


In [68]:
def get_relation(edge):
    u, v, key, attr = edge
    
    # If the key is our custom 'rev_...' string, that IS the relation
    if isinstance(key, str) and key.startswith('rev_'):
        return key 
    
    # Otherwise, check attributes
    relation = attr.get('relation') or attr.get('rel') or attr.get('type') or (key if isinstance(key, str) else None)
    
    return relation

def normalize_kg(kg: nx.MultiDiGraph):
    new_kg = nx.MultiDiGraph()
    
    # 1. Standardize Nodes
    for node, data in kg.nodes(data=True):
        # Infer the label: look for 'label' or 'type', fallback to 'Unknown'
        label = data.get('label') or data.get('type') or 'Unknown'
        
        # Keep everything EXCEPT the old 'label'/'type' keys
        new_data = {k: v for k, v in data.items() if k not in ['label', 'type']}
        new_data['label'] = label
        new_kg.add_node(node, **new_data)
        
    # 2. Standardize Edges
    for u, v, key, attrs in kg.edges(data=True, keys=True):
        # Extract the relation name from existing messy keys/attrs
        if isinstance(key, str) and key not in ['0', '1', '2']: # Check if key is a name
            relation = key
        else:
            relation = attrs.get('relation') or attrs.get('rel') or attrs.get('type') or 'unknown'
        
        excluded = {'relation', 'rel', 'type'}
        new_attrs = {k: v for k, v in attrs.items() if k not in excluded}
        
        # Add the standardized 'relation' key
        new_attrs['relation'] = relation
        
        # Add to new graph
        new_kg.add_edge(u, v, key=relation, **new_attrs)
        
    return new_kg

## Add reverse edges
+ PPI-new kg has no node type
+ need to add reverse edges to the other KGs

In [70]:

def add_reverse_edges(kg: nx.MultiDiGraph):
    reversed_kg = copy.deepcopy(kg)
    
    # Track sizes explicitly
    initial_edge_count = reversed_kg.number_of_edges()
    added_rev_count = 0
    
    for u, v, key, attrs in list(kg.edges(data=True, keys=True)):
        relation = get_relation((u, v, key, attrs))
        if v == u or relation == 'similar' or (relation and relation.startswith('rev_')):
            continue

        rev_rel = f"rev_{relation}"
        
        # SAFE CHECK: Use .has_edge() first
        has_rev = False
        if reversed_kg.has_edge(v, u):
            # Now it is safe to access the edge dictionary
            if rev_rel in reversed_kg[v][u]:
                has_rev = True
        
        if not has_rev:
            new_attrs = {k: v for k, v in attrs.items() if k not in ['type','rel','relation']}
            new_attrs['relation'] = rev_rel
            reversed_kg.add_edge(v, u, key=rev_rel, **new_attrs)
            added_rev_count += 1
            
    final_edge_count = reversed_kg.number_of_edges()
    print(f"Initial: {initial_edge_count}, Final: {final_edge_count}")
    print(f"Diff: {final_edge_count - initial_edge_count}, Logged Adds: {added_rev_count}")
    
    return reversed_kg

In [52]:
test_adkg = add_reverse_edges(ppi_new)

Initial: 382526, Final: 741540
Diff: 359014, Logged Adds: 359014


In [53]:
test_adkg2 = add_reverse_edges(test_adkg)

Initial: 741540, Final: 741540
Diff: 0, Logged Adds: 0


In [71]:
base_kgs = [ad_kg, healthy_kg, ppi_old, ppi_new, prime_kg]
kg_names = ['ad_kg', 'healthy_kg', 'ppi_old', 'ppi_new', 'prime_kg']

for i, kg in enumerate(base_kgs):
    normalized_kg = normalize_kg(kg)
    save_path = f"../datasets/base_kgs/{kg_names[i]}_normalized.pkl"
    save_graph(save_path, normalized_kg)

    print(f"Adding reverse edges to {kg_names[i]}")
    kg_reversed = add_reverse_edges(normalized_kg)
    save_path = f"../datasets/base_kgs/{kg_names[i]}_with_reverse_edges.pkl"
    save_graph(save_path, kg_reversed)
    print()


Save graph with 3732 nodes and 10554 edges to ../datasets/base_kgs/ad_kg_normalized.pkl
Adding reverse edges to ad_kg
Initial: 10554, Final: 10554
Diff: 0, Logged Adds: 0
Save graph with 3732 nodes and 10554 edges to ../datasets/base_kgs/ad_kg_with_reverse_edges.pkl

Save graph with 4161 nodes and 6956 edges to ../datasets/base_kgs/healthy_kg_normalized.pkl
Adding reverse edges to healthy_kg
Initial: 6956, Final: 13911
Diff: 6955, Logged Adds: 6955
Save graph with 4161 nodes and 13911 edges to ../datasets/base_kgs/healthy_kg_with_reverse_edges.pkl

Save graph with 8910 nodes and 209730 edges to ../datasets/base_kgs/ppi_old_normalized.pkl
Adding reverse edges to ppi_old
Initial: 209730, Final: 417125
Diff: 207395, Logged Adds: 207395
Save graph with 8910 nodes and 417125 edges to ../datasets/base_kgs/ppi_old_with_reverse_edges.pkl

Save graph with 17042 nodes and 361928 edges to ../datasets/base_kgs/ppi_new_normalized.pkl
Adding reverse edges to ppi_new
Initial: 361928, Final: 720942
Di

In [72]:
for edge in kg_reversed.edges(data=True, keys=True):
    print(edge)

('AHRR', 'Cadmium', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('AHRR', 'Cadmium', 'rev_INTERACTS_WITH', {'relation': 'rev_INTERACTS_WITH'})
('AHRR', 'Particulate Matter', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('AHRR', 'Particulate Matter', 'rev_INTERACTS_WITH', {'relation': 'rev_INTERACTS_WITH'})
('AHRR', 'ESR1', 'PPI', {'relation': 'PPI'})
('AHRR', 'ESR1', 'rev_PPI', {'relation': 'rev_PPI'})
('Cadmium', 'AHRR', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('Cadmium', 'AHRR', 'rev_INTERACTS_WITH', {'relation': 'rev_INTERACTS_WITH'})
('Cadmium', 'F2RL3', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('Cadmium', 'F2RL3', 'rev_INTERACTS_WITH', {'relation': 'rev_INTERACTS_WITH'})
('Cadmium', 'PRSS23', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('Cadmium', 'PRSS23', 'rev_INTERACTS_WITH', {'relation': 'rev_INTERACTS_WITH'})
('Cadmium', 'RARA', 'INTERACTS_WITH', {'relation': 'INTERACTS_WITH'})
('Cadmium', 'RARA', 'rev_INTERACTS_WITH', {'relation': 'rev_INTE

## Conversion to HeteroData

In [76]:
import torch
from torch_geometric.data import HeteroData

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [77]:
def convert_to_hetero_data(G: nx.MultiDiGraph):
    """
    Converts the hybrid Patient-Protein network into a PyTorch Geometric HeteroData object.
    Initializes features for ALL node types to match Patient feature dimensions.
    """
    print("Starting conversion from NetworkX to HeteroData...")
    data = HeteroData()
    node_mappings = {} # {node_type: {node_name: integer_index}}
    
    # 1. Categorize Nodes and Map Indices
    for node, attrs in G.nodes(data=True):
        n_type = attrs.get('type') or attrs.get('label')
        
        if n_type not in node_mappings:
            node_mappings[n_type] = {}
        
        if node not in node_mappings[n_type]:
            node_mappings[n_type][node] = len(node_mappings[n_type])

    # 2. Process Patient Data (The "Reference" Features)
    p_map = node_mappings.get('Patient', {})
    if not p_map:
        raise ValueError("No 'Patient' nodes found in the graph to use as a feature dimension reference.")
    
    p_ids = sorted(p_map, key=p_map.get)
    patient_x = torch.tensor([G.nodes[pid]['x'] for pid in p_ids], dtype=torch.float)
    
    # Capture the dimension D (e.g., number of genes/features)
    feature_dim = patient_x.size(1) 
    
    data['Patient'].x = patient_x
    data['Patient'].y = torch.tensor([G.nodes[pid]['y'] for pid in p_ids], dtype=torch.long)
    data['Patient'].train_mask = torch.tensor([G.nodes[pid]['train_mask'] for pid in p_ids], dtype=torch.bool)
    data['Patient'].val_mask = torch.tensor([G.nodes[pid]['val_mask'] for pid in p_ids], dtype=torch.bool)
    data['Patient'].test_mask = torch.tensor([G.nodes[pid]['test_mask'] for pid in p_ids], dtype=torch.bool)

    # 3. Process ALL Node Types (Initialize x for all types)
    for n_type, mapping in node_mappings.items():
        if n_type == 'Patient':
            continue  # Already handled above
            
        num_nodes = len(mapping)
        data[n_type].num_nodes = num_nodes
        
        # Initialize features to match Patient feature dimension: use torch.zeros or torch.randn. 
        data[n_type].x = torch.zeros((num_nodes, feature_dim), dtype=torch.float)
        #print(f"Initialized {n_type} nodes: {num_nodes} nodes with feature dim {feature_dim}")

    # 4. Process Edges with separation
    static_edges = {}
    dynamic_edges = {}

    for u, v, r, attrs in G.edges(keys=True, data=True):
        u_type = G.nodes[u].get('type') or G.nodes[u].get('label')
        v_type = G.nodes[v].get('type') or G.nodes[v].get('label')
        if not isinstance(r, str):
            rel = attrs.get('relation') or attrs.get('rel') or attrs.get('type')
        else:
            rel = r
        
        # Replace double underscores with single ones to satisfy PyG requirements
        safe_rel = str(rel).replace('__', '_')
        
        edge_type = (u_type, safe_rel, v_type)
        edge_type = (u_type, safe_rel, v_type)
        
        # Categorize based on node types
        if u_type == 'Patient' or v_type == 'Patient':
            target_dict = dynamic_edges
        else:
            target_dict = static_edges
            
        if edge_type not in target_dict:
            target_dict[edge_type] = []
        
        u_idx = node_mappings[u_type][u]
        v_idx = node_mappings[v_type][v]
        target_dict[edge_type].append([u_idx, v_idx])

    # Finalize Edges in HeteroData
    for etype, content in {**static_edges, **dynamic_edges}.items():
        data[etype].edge_index = torch.tensor(content, dtype=torch.long).t().contiguous()
    
    # 5. Attach the dict to the data object for easy access
    data.static_edge_types = list(static_edges.keys())
    data.dynamic_edge_types = list(dynamic_edges.keys())
    
    print(f"HeteroData created: {len(data.node_types)} node types, {len(data.static_edge_types) + len(data.dynamic_edge_types)} edge types.")
    return data, node_mappings


In [73]:
ad_merge = load_graph("../datasets/ADNI_KGs/ecdf_5/AD_KG/G_adni_merge_ecdf.pkl")
ppi_new_merge = load_graph("../datasets/ADNI_KGs/ecdf_5/PPI_new/G_adni_merge_ecdf.pkl")
ppi_old_merge = load_graph("../datasets/ADNI_KGs/ecdf_5/PPI_old/G_adni_merge_ecdf.pkl")
prime_merge = load_graph("../datasets/ADNI_KGs/ecdf_5/Prime_KG/G_adni_merge_ecdf.pkl")

Load graph with 8564 nodes and 203351 edges
Load graph with 21947 nodes and 1889847 edges
Load graph with 13815 nodes and 1719872 edges
Load graph with 9814 nodes and 772207 edges


In [78]:
merge_kgs = [ad_merge, ppi_new_merge, ppi_old_merge, prime_merge]

for kg in merge_kgs:
    data, node_mappings = convert_to_hetero_data(kg)

Starting conversion from NetworkX to HeteroData...


/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_9619/732319593.py:26: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/torch/csrc/utils/tensor_new.cpp:277.)
  patient_x = torch.tensor([G.nodes[pid]['x'] for pid in p_ids], dtype=torch.float)


HeteroData created: 25 node types, 866 edge types.
Starting conversion from NetworkX to HeteroData...
HeteroData created: 23 node types, 550 edge types.
Starting conversion from NetworkX to HeteroData...
HeteroData created: 23 node types, 552 edge types.
Starting conversion from NetworkX to HeteroData...
HeteroData created: 28 node types, 600 edge types.
